In [ ]:
import os
from pathlib import Path
import random
import shutil
import json

# Kaggle credentials
os.environ["KAGGLE_USERNAME"] = "olabrenne"
os.environ["KAGGLE_KEY"] = "KGAT_7415f001333b54270ac6407f767aac7a"

print("Kaggle API keys loaded.")

!pip install -q kaggle scikit-learn seaborn

# Download dataset
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content --unzip

# Verify folders
!ls -la /content
!find /content -maxdepth 2 -type d \( -name "Training" -o -name "Testing" \)

In [ ]:

# 2) Build train / val / test directory structure
random.seed(42)

src_train = Path("/content/Training")
src_test = Path("/content/Testing")

dst_root = Path("/content/data")
dst_train = dst_root / "train"
dst_val = dst_root / "val"
dst_test = dst_root / "test"

if dst_root.exists():
    shutil.rmtree(dst_root)

# Copy original test set directly
shutil.copytree(src_test, dst_test)

# Create train/val split from original training set
VAL_FRAC = 0.15
classes = sorted([d.name for d in src_train.iterdir() if d.is_dir()])
print("Classes found:", classes)

for c in classes:
    imgs = [p for p in (src_train / c).glob("*") if p.is_file()]
    random.shuffle(imgs)

    n_val = int(len(imgs) * VAL_FRAC)

    (dst_train / c).mkdir(parents=True, exist_ok=True)
    (dst_val / c).mkdir(parents=True, exist_ok=True)

    for p in imgs[:n_val]:
        shutil.copy2(p, dst_val / c / p.name)

    for p in imgs[n_val:]:
        shutil.copy2(p, dst_train / c / p.name)

print("Split complete.")

# Show class distribution
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    base = dst_root / split
    for c in sorted([d.name for d in base.iterdir() if d.is_dir()]):
        print(c, ":", len(list((base / c).glob("*"))))

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, f1_score

tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 4) Config
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 30
INITIAL_LR = 1e-5
UNFREEZE_LAST_N = 50
DROPOUT_RATE = 0.3

train_dir = str(dst_train)
val_dir = str(dst_val)
test_dir = str(dst_test)

In [ ]:

# 5) Load datasets

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

Found 4760 files belonging to 4 classes.
Found 840 files belonging to 4 classes.
Found 1600 files belonging to 4 classes.
Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']
Number of classes: 4


In [ ]:
# 6) Data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.2),
], name="data_augmentation")

In [ ]:
# 7) Build ResNet50 model
base = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)

base.trainable = True
for layer in base.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.resnet.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0, label_smoothing=0.05),
    metrics=["accuracy", tf.keras.metrics.F1Score(average="macro", name="f1_macro")]
)
model.summary()

In [ ]:

# 8) Callbacks

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "resnet50_best.keras",
        monitor="val_f1_macro",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_f1_macro",
        patience=5,
        restore_best_weights=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1
    )
]

# 9) Train model

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# 9) Evaluation on Test Set
best = tf.keras.models.load_model("resnet50_best.keras")
test_loss, test_acc, test_f1 = best.evaluate(test_ds)

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds], axis=0)
y_pred = np.argmax(best.predict(test_ds), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:

# 11) Predictions and evaluation metrics

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds], axis=0)
y_pred_probs = best.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)

# 12) Plot confusion matrix

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# 13) Plot training history

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Val loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 3, 2)
plt.plot(history.history["accuracy"], label="Train acc")
plt.plot(history.history["val_accuracy"], label="Val acc")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 3, 3)
plt.plot(history.history["f1_macro"], label="Train F1")
plt.plot(history.history["val_f1_macro"], label="Val F1")
plt.legend()
plt.title("Macro F1")

plt.tight_layout()
plt.show()

In [ ]:
files.download("resnet50_best.keras")
